In [ ]:
!pip install -U ultralytics roboflow pyyaml pandas

In [ ]:
# =========================
# Project Config
# =========================
# Roboflow 프로젝트 정보
ROBOFLOW_WORKSPACE = "2026-oss"
ROBOFLOW_PROJECT = "ai-picture-book-object-detection"

# Roboflow v14 데이터셋 버전
# - v13에서 인식이 불안정했던 tactile_flower, tactile_flowerpot 데이터를 보완한 버전
ROBOFLOW_VERSION = 14

# 사용할 YOLO 모델과 학습 run 이름
MODEL_NAME = "yolo11n.pt"
RUN_NAME = f"picture_book_yolo11n_v{ROBOFLOW_VERSION}"

# 학습 설정
IMG_SIZE = 640
EPOCHS = 150
BATCH_SIZE = 64

# 평가/예측 설정
# - 학습 자체가 아니라 test image prediction 단계에서 사용
# - 가까운 객체가 동시에 탐지될 수 있도록 prediction 단계에서는 conf를 조금 낮추고 iou를 높게 둔다.
CONF_THRESHOLD = 0.25
PREDICT_CONF_THRESHOLD = 0.15
PREDICT_IOU_THRESHOLD = 0.80
MAX_DETECTIONS = 50

In [ ]:
# Colab에서 GPU가 정상적으로 연결되었는지 확인
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    raise RuntimeError("GPU가 연결되어 있지 않습니다. Colab 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.")

In [ ]:
# Roboflow API Key를 Colab Secrets에서 불러오고, 없으면 직접 입력받음
from getpass import getpass

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

    # 기존 Colab Secret 이름을 ROBOFLOW로 저장한 경우도 처리
    if not ROBOFLOW_API_KEY:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Roboflow Private API Key를 입력하세요: ")

ROBOFLOW_API_KEY = ROBOFLOW_API_KEY.strip()

print("Roboflow API Key loaded:", len(ROBOFLOW_API_KEY) > 0)
print("API Key length:", len(ROBOFLOW_API_KEY))

In [ ]:
# Roboflow에서 v14 데이터셋 다운로드
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

# "yolov8"은 모델명이 아니라 데이터셋 export format임
# 실제 학습 모델은 아래에서 YOLO("yolo11n.pt")로 지정함
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

print("Dataset version:", ROBOFLOW_VERSION)
print("Dataset downloaded to:", dataset.location)

In [ ]:
# data.yaml 파일을 확인하고, 학습 경로를 절대경로로 보정
import os
import yaml

yaml_path = os.path.join(dataset.location, "data.yaml")

print("Original data.yaml path:", yaml_path)

with open(yaml_path, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

print("Original data.yaml:")
print(data_yaml)

# 경로 오류를 줄이기 위해 train/valid/test 이미지 경로를 절대경로로 수정
data_yaml["train"] = os.path.join(dataset.location, "train", "images")
data_yaml["val"] = os.path.join(dataset.location, "valid", "images")
data_yaml["test"] = os.path.join(dataset.location, "test", "images")

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print("\nUpdated data.yaml:")
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# v14 데이터셋에 10개 클래스가 모두 포함되어 있는지 확인

EXPECTED_CLASSES = [
    "book_flower",
    "book_flowerpot",
    "book_monkey",
    "book_stone",
    "braille",
    "tactile_flower",
    "tactile_flowerpot",
    "tactile_monkey",
    "tactile_stone",
    "text",
]

with open(yaml_path, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

names = data_yaml.get("names")

# names가 dict 또는 list 형태일 수 있으므로 둘 다 처리
if isinstance(names, dict):
    actual_classes = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
else:
    actual_classes = names

print("Actual classes:")
for idx, name in enumerate(actual_classes):
    print(f"{idx}: {name}")

print("\nExpected class count:", len(EXPECTED_CLASSES))
print("Actual class count:", len(actual_classes))

missing_classes = sorted(set(EXPECTED_CLASSES) - set(actual_classes))
extra_classes = sorted(set(actual_classes) - set(EXPECTED_CLASSES))

print("\nMissing classes:", missing_classes)
print("Extra classes:", extra_classes)

assert len(actual_classes) == len(EXPECTED_CLASSES), "클래스 개수가 예상과 다릅니다."
assert set(actual_classes) == set(EXPECTED_CLASSES), "클래스 목록이 예상과 다릅니다."

print("\n클래스 검증 완료: v14 데이터셋에 10개 클래스가 모두 포함되어 있습니다.")

In [ ]:
# train, valid, test 데이터가 정상적으로 나뉘어 있는지 확인
import glob

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(dataset.location, split, "images")
    label_dir = os.path.join(dataset.location, split, "labels")

    image_count = len(glob.glob(os.path.join(image_dir, "*")))
    label_count = len(glob.glob(os.path.join(label_dir, "*")))

    print(f"{split}: images={image_count}, labels={label_count}")

In [ ]:
# YOLO11n 모델을 불러와 v14 데이터셋으로 학습
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=yaml_path,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    name=RUN_NAME,
    project="runs/detect",
    device=0,
    seed=42,
    deterministic=True,
    patience=30,
    cos_lr=True,
    cache=True
)

In [ ]:
# YOLO11n v14 학습 결과의 best.pt 경로 확인
from pathlib import Path

best_weights = [
    path for path in Path.cwd().rglob("best.pt")
    if RUN_NAME in str(path)
]

best_weights = sorted(best_weights)

print("best.pt files:")
for path in best_weights:
    print(path)

assert len(best_weights) > 0, "YOLO11n v14 best.pt 파일을 찾지 못했습니다."

best_model_path = str(best_weights[-1])

print("\nBest model path:", best_model_path)

In [ ]:
# best.pt 모델을 test set 기준으로 평가
best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=yaml_path,
    split="test",
    imgsz=IMG_SIZE
)

In [ ]:
# 전체 성능 지표 정리
summary = {
    "model": MODEL_NAME,
    "dataset_version": ROBOFLOW_VERSION,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP@0.5": float(metrics.box.map50),
    "mAP@0.5:0.95": float(metrics.box.map),
    "best_model_path": best_model_path,
}

summary

In [ ]:
# 클래스별 mAP 성능 확인
import pandas as pd

class_result_rows = []

ap50_values = getattr(metrics.box, "ap50", None)
ap_values = getattr(metrics.box, "ap", None)

for idx, class_name in enumerate(actual_classes):
    row = {
        "class_id": idx,
        "class_name": class_name,
        "mAP@0.5": float(ap50_values[idx]) if ap50_values is not None and idx < len(ap50_values) else None,
        "mAP@0.5:0.95": float(ap_values[idx]) if ap_values is not None and idx < len(ap_values) else None,
    }
    class_result_rows.append(row)

class_results_df = pd.DataFrame(class_result_rows)
class_results_df

In [ ]:
# 평가 결과를 JSON, CSV 파일로 저장
import json

os.makedirs("results", exist_ok=True)

summary_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_summary.json"
class_result_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_class_results.csv"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

class_results_df.to_csv(class_result_path, index=False, encoding="utf-8-sig")

print("Saved summary:", summary_path)
print("Saved class results:", class_result_path)

In [ ]:
# test 이미지에 대해 실제 예측 결과 저장
# 가까운 객체가 동시에 잡히는지 확인하기 위해 prediction 단계에서는
# confidence threshold를 낮추고 NMS IoU threshold를 높게 설정한다.
test_image_path = os.path.join(dataset.location, "test", "images")

predict_results = best_model.predict(
    source=test_image_path,
    conf=PREDICT_CONF_THRESHOLD,
    iou=PREDICT_IOU_THRESHOLD,
    max_det=MAX_DETECTIONS,
    save=True,
    name=f"{RUN_NAME}_test_predictions"
)

print("Prediction settings")
print("conf:", PREDICT_CONF_THRESHOLD)
print("iou:", PREDICT_IOU_THRESHOLD)
print("max_det:", MAX_DETECTIONS)

In [ ]:
# 저장된 예측 결과 이미지 일부 확인
from IPython.display import Image, display

result_images = sorted(glob.glob(f"runs/detect/{RUN_NAME}_test_predictions*/*.jpg"))

print("Detected images:", len(result_images))

for img_path in result_images[:10]:
    display(Image(filename=img_path))

In [ ]:
# 성능 결과를 그래프로 시각화하여 저장
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

os.makedirs("results", exist_ok=True)
GRAPH_DIR = "results/graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)

graph_files = []

# 1. 전체 성능 지표 bar chart
metrics_plot_df = pd.DataFrame([
    {"metric": "Precision", "score": summary["precision"]},
    {"metric": "Recall", "score": summary["recall"]},
    {"metric": "mAP@0.5", "score": summary["mAP@0.5"]},
    {"metric": "mAP@0.5:0.95", "score": summary["mAP@0.5:0.95"]},
])

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics_plot_df["metric"], metrics_plot_df["score"])
plt.ylim(0, 1.0)
plt.title(f"YOLO11n v{ROBOFLOW_VERSION} Overall Metrics")
plt.ylabel("Score")
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f"{height:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.tight_layout()
overall_graph_path = f"{GRAPH_DIR}/yolo11n_v{ROBOFLOW_VERSION}_overall_metrics.png"
plt.savefig(overall_graph_path, dpi=200)
plt.show()
graph_files.append(overall_graph_path)

# 2. 클래스별 mAP@0.5 horizontal bar chart
class_map50_df = class_results_df.sort_values("mAP@0.5", ascending=True)

plt.figure(figsize=(10, max(5, len(class_map50_df) * 0.45)))
plt.barh(class_map50_df["class_name"], class_map50_df["mAP@0.5"])
plt.xlim(0, 1.0)
plt.title(f"YOLO11n v{ROBOFLOW_VERSION} Class-wise mAP@0.5")
plt.xlabel("mAP@0.5")
plt.grid(axis="x", linestyle="--", alpha=0.4)

for idx, value in enumerate(class_map50_df["mAP@0.5"]):
    if pd.notna(value):
        plt.text(value + 0.01, idx, f"{value:.3f}", va="center", fontsize=9)

plt.tight_layout()
class_map50_graph_path = f"{GRAPH_DIR}/yolo11n_v{ROBOFLOW_VERSION}_class_map50.png"
plt.savefig(class_map50_graph_path, dpi=200)
plt.show()
graph_files.append(class_map50_graph_path)

# 3. 클래스별 mAP@0.5:0.95 horizontal bar chart
class_map_df = class_results_df.sort_values("mAP@0.5:0.95", ascending=True)

plt.figure(figsize=(10, max(5, len(class_map_df) * 0.45)))
plt.barh(class_map_df["class_name"], class_map_df["mAP@0.5:0.95"])
plt.xlim(0, 1.0)
plt.title(f"YOLO11n v{ROBOFLOW_VERSION} Class-wise mAP@0.5:0.95")
plt.xlabel("mAP@0.5:0.95")
plt.grid(axis="x", linestyle="--", alpha=0.4)

for idx, value in enumerate(class_map_df["mAP@0.5:0.95"]):
    if pd.notna(value):
        plt.text(value + 0.01, idx, f"{value:.3f}", va="center", fontsize=9)

plt.tight_layout()
class_map_graph_path = f"{GRAPH_DIR}/yolo11n_v{ROBOFLOW_VERSION}_class_map5095.png"
plt.savefig(class_map_graph_path, dpi=200)
plt.show()
graph_files.append(class_map_graph_path)

print("Saved graph files:")
for path in graph_files:
    print(path)


In [ ]:
# 학습 과정 그래프 저장
# Ultralytics가 생성한 results.csv를 이용해 epoch별 metric/loss 변화를 그래프로 저장한다.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

train_result_candidates = [
    path for path in Path.cwd().rglob("results.csv")
    if RUN_NAME in str(path)
]

print("Found training results.csv files:")
for path in train_result_candidates:
    print(path)

if len(train_result_candidates) > 0:
    train_results_csv = sorted(train_result_candidates)[-1]
    train_df = pd.read_csv(train_results_csv)
    train_df.columns = [col.strip() for col in train_df.columns]

    # epoch column이 없으면 index를 epoch처럼 사용
    x_values = train_df["epoch"] if "epoch" in train_df.columns else range(1, len(train_df) + 1)

    metric_candidates = [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    ]
    available_metrics = [col for col in metric_candidates if col in train_df.columns]

    if available_metrics:
        plt.figure(figsize=(10, 6))
        for col in available_metrics:
            plt.plot(x_values, train_df[col], label=col.replace("metrics/", ""))
        plt.ylim(0, 1.0)
        plt.title(f"YOLO11n v{ROBOFLOW_VERSION} Training Metrics by Epoch")
        plt.xlabel("Epoch")
        plt.ylabel("Score")
        plt.grid(True, linestyle="--", alpha=0.4)
        plt.legend()
        plt.tight_layout()
        training_metric_graph_path = f"{GRAPH_DIR}/yolo11n_v{ROBOFLOW_VERSION}_training_metrics.png"
        plt.savefig(training_metric_graph_path, dpi=200)
        plt.show()
        graph_files.append(training_metric_graph_path)

    loss_candidates = [
        "train/box_loss",
        "train/cls_loss",
        "train/dfl_loss",
        "val/box_loss",
        "val/cls_loss",
        "val/dfl_loss",
    ]
    available_losses = [col for col in loss_candidates if col in train_df.columns]

    if available_losses:
        plt.figure(figsize=(10, 6))
        for col in available_losses:
            plt.plot(x_values, train_df[col], label=col)
        plt.title(f"YOLO11n v{ROBOFLOW_VERSION} Loss Curves by Epoch")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(True, linestyle="--", alpha=0.4)
        plt.legend()
        plt.tight_layout()
        training_loss_graph_path = f"{GRAPH_DIR}/yolo11n_v{ROBOFLOW_VERSION}_training_losses.png"
        plt.savefig(training_loss_graph_path, dpi=200)
        plt.show()
        graph_files.append(training_loss_graph_path)
else:
    print("학습 results.csv를 찾지 못했습니다. 학습 완료 후 다시 실행하세요.")

print("Current graph files:")
for path in graph_files:
    print(path)


In [ ]:
# Ultralytics가 자동 생성한 성능 시각화 파일 복사
# confusion matrix, PR curve, F1 curve 등 자동 생성 이미지가 있으면 results/graphs로 복사한다.
from pathlib import Path
import shutil

run_dir = Path(best_model_path).parents[1]  # .../weights/best.pt -> run directory
print("Run directory:", run_dir)

auto_plot_files = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "PR_curve.png",
    "labels.jpg",
]

copied_auto_plots = []

for file_name in auto_plot_files:
    src_path = run_dir / file_name
    if src_path.exists():
        dst_path = Path(GRAPH_DIR) / f"yolo11n_v{ROBOFLOW_VERSION}_{file_name}"
        shutil.copy(src_path, dst_path)
        copied_auto_plots.append(str(dst_path))
        if str(dst_path) not in graph_files:
            graph_files.append(str(dst_path))

print("Copied auto plot files:")
for path in copied_auto_plots:
    print(path)


In [ ]:
# 모델 파일과 학습 설정 파일을 GitHub 업로드용 경로로 복사
import os
import shutil
from pathlib import Path

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

final_model_path = f"models/yolo11n_v{ROBOFLOW_VERSION}_best.pt"
shutil.copy(best_model_path, final_model_path)
print("Saved final model:", final_model_path)

run_dir = Path(best_model_path).parents[1]
args_src_path = run_dir / "args.yaml"
args_dst_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_args.yaml"

if args_src_path.exists():
    shutil.copy(args_src_path, args_dst_path)
    print("Saved args.yaml:", args_dst_path)
else:
    print("args.yaml 파일을 찾지 못했습니다:", args_src_path)


In [ ]:
# v13 결과 파일이 함께 있는 경우, v13과 v14 성능 비교 파일을 생성한다.
# - Colab을 새로 시작한 경우 v13 파일이 없을 수 있으므로, 없으면 자동으로 건너뛴다.
# - v13 파일을 비교하려면 아래 두 파일을 results/ 폴더에 업로드한 뒤 실행하면 된다.
#   1) results/yolo11n_v13_summary.json
#   2) results/yolo11n_v13_class_results.csv

import os
import json
import pandas as pd
import matplotlib.pyplot as plt

comparison_files = []

v13_summary_path = "results/yolo11n_v13_summary.json"
v13_class_result_path = "results/yolo11n_v13_class_results.csv"

v14_summary_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_summary.json"
v14_class_result_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_class_results.csv"

if os.path.exists(v13_summary_path) and os.path.exists(v13_class_result_path):
    with open(v13_summary_path, "r", encoding="utf-8") as f:
        v13_summary = json.load(f)

    with open(v14_summary_path, "r", encoding="utf-8") as f:
        v14_summary = json.load(f)

    v13_class_df = pd.read_csv(v13_class_result_path)
    v14_class_df = pd.read_csv(v14_class_result_path)

    # 1. 전체 성능 비교 CSV 저장
    overall_comparison_df = pd.DataFrame([
        {
            "version": "v13",
            "dataset_version": v13_summary.get("dataset_version", 13),
            "precision": v13_summary.get("precision"),
            "recall": v13_summary.get("recall"),
            "mAP@0.5": v13_summary.get("mAP@0.5"),
            "mAP@0.5:0.95": v13_summary.get("mAP@0.5:0.95"),
        },
        {
            "version": f"v{ROBOFLOW_VERSION}",
            "dataset_version": v14_summary.get("dataset_version", ROBOFLOW_VERSION),
            "precision": v14_summary.get("precision"),
            "recall": v14_summary.get("recall"),
            "mAP@0.5": v14_summary.get("mAP@0.5"),
            "mAP@0.5:0.95": v14_summary.get("mAP@0.5:0.95"),
        },
    ])

    overall_comparison_path = "results/yolo11n_v13_v14_comparison.csv"
    overall_comparison_df.to_csv(overall_comparison_path, index=False, encoding="utf-8-sig")
    comparison_files.append(overall_comparison_path)

    # 2. 클래스별 비교 CSV 저장
    class_comparison_df = v13_class_df.merge(
        v14_class_df,
        on=["class_id", "class_name"],
        how="outer",
        suffixes=("_v13", "_v14")
    )

    for metric in ["mAP@0.5", "mAP@0.5:0.95"]:
        class_comparison_df[f"{metric}_diff_v14_minus_v13"] = (
            class_comparison_df[f"{metric}_v14"] - class_comparison_df[f"{metric}_v13"]
        )

    class_comparison_path = "results/yolo11n_v13_v14_class_comparison.csv"
    class_comparison_df.to_csv(class_comparison_path, index=False, encoding="utf-8-sig")
    comparison_files.append(class_comparison_path)

    # 3. 전체 성능 비교 그래프 저장
    metric_columns = ["precision", "recall", "mAP@0.5", "mAP@0.5:0.95"]
    plot_df = overall_comparison_df.set_index("version")[metric_columns].T

    plt.figure(figsize=(9, 5))
    plot_df.plot(kind="bar", ax=plt.gca())
    plt.ylim(0, 1.0)
    plt.title("YOLO11n v13 vs v14 Overall Metrics")
    plt.ylabel("Score")
    plt.xticks(rotation=0)
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()

    overall_graph_path = "results/graphs/yolo11n_v13_v14_overall_comparison.png"
    plt.savefig(overall_graph_path, dpi=200)
    plt.show()
    comparison_files.append(overall_graph_path)
    if "graph_files" in globals() and overall_graph_path not in graph_files:
        graph_files.append(overall_graph_path)

    # 4. tactile 클래스 중점 비교 그래프 저장
    tactile_targets = ["tactile_flower", "tactile_flowerpot"]
    tactile_df = class_comparison_df[class_comparison_df["class_name"].isin(tactile_targets)].copy()

    if not tactile_df.empty:
        tactile_plot_rows = []
        for _, row in tactile_df.iterrows():
            tactile_plot_rows.append({
                "class_name": row["class_name"],
                "version": "v13",
                "mAP@0.5": row.get("mAP@0.5_v13"),
                "mAP@0.5:0.95": row.get("mAP@0.5:0.95_v13"),
            })
            tactile_plot_rows.append({
                "class_name": row["class_name"],
                "version": f"v{ROBOFLOW_VERSION}",
                "mAP@0.5": row.get("mAP@0.5_v14"),
                "mAP@0.5:0.95": row.get("mAP@0.5:0.95_v14"),
            })

        tactile_plot_df = pd.DataFrame(tactile_plot_rows)
        tactile_pivot = tactile_plot_df.pivot(index="class_name", columns="version", values="mAP@0.5")

        plt.figure(figsize=(8, 5))
        tactile_pivot.plot(kind="bar", ax=plt.gca())
        plt.ylim(0, 1.0)
        plt.title("YOLO11n v13 vs v14 Tactile Class mAP@0.5")
        plt.ylabel("mAP@0.5")
        plt.xlabel("Class")
        plt.xticks(rotation=0)
        plt.grid(axis="y", linestyle="--", alpha=0.4)
        plt.tight_layout()

        tactile_graph_path = "results/graphs/yolo11n_v13_v14_tactile_classes_comparison.png"
        plt.savefig(tactile_graph_path, dpi=200)
        plt.show()
        comparison_files.append(tactile_graph_path)
        if "graph_files" in globals() and tactile_graph_path not in graph_files:
            graph_files.append(tactile_graph_path)

    # 5. 비교 리포트 md 저장
    tactile_summary_lines = []
    if not tactile_df.empty:
        for _, row in tactile_df.iterrows():
            tactile_summary_lines.append(
                f"- `{row['class_name']}`: "
                f"mAP@0.5 {row.get('mAP@0.5_v13'):.3f} → {row.get('mAP@0.5_v14'):.3f}, "
                f"mAP@0.5:0.95 {row.get('mAP@0.5:0.95_v13'):.3f} → {row.get('mAP@0.5:0.95_v14'):.3f}"
            )

    report_path = "results/yolo11n_v13_v14_comparison.md"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"""# YOLO11n v13 vs v14 성능 비교 리포트

## 1. 작업 목적

Roboflow v14 데이터셋은 v13에서 인식이 불안정했던 `tactile_flower`, `tactile_flowerpot` 클래스의 데이터를 추가 보완한 버전이다. 본 비교는 v14 학습 결과가 v13 대비 전체 성능 및 tactile 클래스 성능 개선으로 이어졌는지 확인하기 위해 작성하였다.

## 2. 전체 성능 비교

{overall_comparison_df.to_markdown(index=False)}

## 3. 중점 비교 클래스

{chr(10).join(tactile_summary_lines) if tactile_summary_lines else '- tactile 클래스 비교 결과를 생성하지 못했습니다.'}

## 4. 해석 기준

- `tactile_flower`, `tactile_flowerpot`의 mAP@0.5가 상승하면 해당 클래스의 탐지 성공률이 개선된 것으로 볼 수 있다.
- mAP@0.5:0.95가 함께 상승하면 bounding box 위치 정밀도까지 개선된 것으로 볼 수 있다.
- 전체 성능이 유지되거나 상승하면서 tactile 클래스 성능이 개선되면 v14 데이터셋 보완이 유효하다고 판단할 수 있다.
""")
    comparison_files.append(report_path)

    print("Saved v13/v14 comparison files:")
    for path in comparison_files:
        print(path)
else:
    print("v13 summary/class result 파일이 없어 v13/v14 비교 파일 생성을 건너뜁니다.")
    print("비교를 생성하려면 아래 파일을 results/ 폴더에 업로드한 뒤 이 셀을 다시 실행하세요.")
    print("-", v13_summary_path)
    print("-", v13_class_result_path)


In [ ]:
# 백엔드 연동 시 필요한 모델 정보 정리
model_info = {
    "model_name": MODEL_NAME,
    "run_name": RUN_NAME,
    "dataset_version": ROBOFLOW_VERSION,
    "input": {
        "type": "image_or_webcam_frame",
        "image_size": IMG_SIZE,
    },
    "output": {
        "class_name": "detected object class",
        "confidence": "detection confidence score",
        "bbox": "[x1, y1, x2, y2]",
    },
    "classes": actual_classes,
    "best_model_path": best_model_path,
    "final_model_path": final_model_path if "final_model_path" in globals() else None,
    "metrics": summary,
    "visualization_files": graph_files if "graph_files" in globals() else [],
    "comparison_files": comparison_files if "comparison_files" in globals() else [],
    "prediction_settings": {
        "conf": PREDICT_CONF_THRESHOLD,
        "iou": PREDICT_IOU_THRESHOLD,
        "max_det": MAX_DETECTIONS,
    },
    "integration_notes": {
        "finger_recognition": (
            "손가락 인식은 YOLO 학습 노트북이 아니라 Backend/MediaPipe Hands 단계에서 처리한다. "
            "YOLO 모델은 손끝 좌표와 매칭할 객체 bbox를 안정적으로 제공하는 역할이다."
        ),
        "close_object_detection": (
            "가까운 객체가 동시에 탐지되는지 확인하기 위해 prediction 단계에서 conf를 낮추고 "
            "NMS IoU threshold를 높게 설정했다. 데이터셋 라벨링에서는 가까운 객체도 각각 독립 bbox로 라벨링되어야 한다."
        ),
    },
}

model_info_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_model_info.json"

with open(model_info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print("Saved model info:", model_info_path)
model_info

In [ ]:
# GitHub 업로드 및 백업용 zip 파일 생성
# runs/ 전체는 용량이 커질 수 있으므로 models/와 results/만 압축한다.
from google.colab import files

zip_name = f"yolo11n_v{ROBOFLOW_VERSION}_results.zip"
!zip -r {zip_name} models results

files.download(zip_name)
